# Notebook split_data_v01.ipynb: Data Preparation y Partición Cronológica

**Fase CRISP-DM:** Data Preparation
**Proyecto:** Predicción de tráfico en carretera mediante análisis de series temporales (M30)
**Tutor:** Jesús Bonilla

## Objetivo del Notebook
Este cuaderno documenta el preprocesamiento de los datos horarios de intensidad, ocupación y velocidad media provenientes de 4 sensores de la M30. El objetivo principal es transformar un conjunto de datos transaccional (formato *Long*) en una matriz de series temporales multivariante (formato *Wide*), asegurando la integridad cronológica y preparando los subconjuntos físicos para la fase de modelado sin cometer *Data Leakage* (fuga de información).

## Decisiones Metodológicas y Flujo de Trabajo

### 1. Reestructuración Multivariante (Pivotado)
* **Acción:** Transformación del dataset de formato largo a ancho, estableciendo la fecha como índice temporal estricto.
* **Justificación Académica:** Modelos multivariantes espaciotemporales (como VAR o VECM) y arquitecturas Deep Learning requieren que la información de todos los sensores en el instante $t$ esté alineada en la misma fila para poder capturar las correlaciones cruzadas simultáneas.

### 2. Integridad Temporal estricta
* **Acción:** Imposición de una frecuencia horaria constante mediante el remuestreo del índice temporal.
* **Justificación Académica:** Los modelos estadísticos clásicos (ARIMA, SARIMAX, VAR) asumen matemáticamente una equidistancia temporal perfecta entre observaciones. Esta acción revela las franjas horarias donde ningún sensor emitió datos (caídas del sistema).

### 3. Estrategia de Imputación Conservadora
* **Acción:** Aplicación de propagación hacia adelante (*Forward Fill*) con un límite estricto de 6 horas.
* **Justificación Académica:** El uso de `ffill` asegura que solo se utiliza información pasada ($t-1$) para imputar datos en $t$, evitando el *Data Leakage* que generarían técnicas como la interpolación lineal o el rellenado hacia atrás (*backward fill*). El límite de 6 horas previene la creación de tendencias artificiales en apagones prolongados del sensor.

### 4. Feature Engineering Temporal
* **Acción:** Extracción de características temporales (hora, día de la semana, mes, indicador de fin de semana) directamente del índice.
* **Justificación Académica:** Las variables temporales, especialmente el flag booleano de "fin de semana", son los predictores exógenos más importantes para los algoritmos de Machine Learning basados en árboles (Random Forest, XGBoost), dada la fuerte estacionalidad semanal del flujo de tráfico.

### 5. Partición Física (Hold-Out Cronológico)
* **Acción:** Separación estricta del dataset en tres conjuntos:
  * **Train (Entrenamiento):** Enero 2024 – Octubre 2025 (22 meses)
  * **Validation (Validación):** Noviembre 2025 – Diciembre 2025 (2 meses)
  * **Test (Evaluación):** Enero 2026 – Febrero 2026 (2 meses)
* **Justificación Académica:** En series temporales es metodológicamente incorrecto realizar una partición aleatoria (*shuffle*). La división puramente cronológica garantiza que los modelos se entrenen exclusivamente con el pasado para predecir el futuro. La inclusión de un conjunto de Validación es obligatoria para el ajuste de hiperparámetros (ej. *Early Stopping* en LSTM) sin contaminar el conjunto de Test.

## Archivos de Salida
La ejecución de este notebook genera tres ficheros CSV listos para ser consumidos en la fase de **Modeling**:
* `data_train.csv`
* `data_val.csv`
* `data_test.csv`
*(Nota: Estos archivos están excluidos del control de versiones mediante .gitignore).*

# Paso 1: Carga y Estructuración Básica

In [ ]:
import pandas as pd

print("Ejecutando Paso 1...")

# Hacemos carga del dataset
df = pd.read_csv("../data/processed/dataset_m30_4_sensores_final.csv", sep=";")

# Hacemos conversion a fecha de datetime de Pandas
df['fecha'] = pd.to_datetime(df['fecha'])

# Ordenamos por id y fecha, garantizando que el dataset esta coorrectamente secuenciado antes de pivotar
df = df.sort_values(['fecha', 'id'])

# Verificaciones de control de calidad
print(f"Total de filas cargadas: {len(df)}")
print(f"Rango temporal: desde {df['fecha'].min()} hasta {df['fecha'].max()}")
print("\nMuestra de los datos (primeras 3 filas):")
display(df.head(3)) 

Ejecutando Paso 1...
Total de filas cargadas: 74746
Rango temporal: desde 2024-01-01 00:00:00 hasta 2026-02-28 23:00:00

Muestra de los datos (primeras 3 filas):


,id,fecha,intensidad,ocupacion,vmed,mes_num,dia_semana_num,hora
0,3820,2024-01-01,1038.25,1.50,90.25,1,0,0
18574,6642,2024-01-01,954.25,1.25,94.50,1,0,0
37460,6676,2024-01-01,1207.25,1.50,90.25,1,0,0


# Paso 2: Pivotado (Formato Wide).

In [2]:
print("Ejecutando Paso 2...")

# Pivotamos: 'fecha' pasa a ser el índice, y cada 'id' genera nuevas columnas
df_pivot = df.pivot(index = 'fecha', columns = 'id', values = ['intensidad', 'ocupacion', 'vmed'])

# Ajustamos la creacion del MultiIdex de Pandas, aplanado de datos
df_pivot.columns = [f'{var}_{sensor}' for var, sensor in df_pivot.columns]

# Nos aseguramos que el indice este estrictamente ordenado 
df_pivot = df_pivot.sort_index()

# Verificaciones de control de calidad
print(f"Dimensiones del dataset tras el pivot: {df_pivot.shape} (Filas, Columnas)")
print("\nMuestra de los datos pivotados (primeras 3 filas):")
display(df_pivot.head(3)) 

Ejecutando Paso 2...
Dimensiones del dataset tras el pivot: (18892, 12) (Filas, Columnas)

Muestra de los datos pivotados (primeras 3 filas):


,intensidad_3820,intensidad_6642,intensidad_6676,intensidad_6782,ocupacion_3820,ocupacion_6642,ocupacion_6676,ocupacion_6782,vmed_3820,vmed_6642,vmed_6676,vmed_6782
fecha,,,,,,,,,,,,
2024-01-01 00:00:00,1038.25,954.25,1207.25,796.00,1.50,1.25,1.50,3.0,90.25,94.50,90.25,98.75
2024-01-01 01:00:00,3916.00,4013.50,4042.25,3702.25,9.25,9.75,8.25,8.0,81.75,87.25,82.25,95.75
2024-01-01 02:00:00,3114.50,3021.00,3287.00,2857.75,7.50,7.00,7.25,4.0,83.00,89.75,83.75,97.25


# Paso 3: Integridad Temporal e Imputación de Nulos

In [3]:
print("Ejecutando Paso 3...")

# 1. Forzar la frecuencia horaria estricta (esto añade las 68 horas que faltan)
df_pivot = df_pivot.asfreq('h')

# 2. Conteo de NaNs ANTES de imputar
nans_previos = df_pivot.isnull().any(axis=1).sum()
print(f"Total de filas con algún NaN ANTES de imputar: {nans_previos} de {len(df_pivot)}")

# 3. Imputación controlada (Forward Fill con limite de 6 horas)
MAX_GAP_HORAS = 6
df_pivot = df_pivot.ffill(limit=MAX_GAP_HORAS)

# 4. Verificacion POST-imputación
nans_posteriores = df_pivot.isnull().any(axis=1).sum()
print(f"Total de filas con algun NaN TRAS imputar (limite {MAX_GAP_HORAS}h): {nans_posteriores}")

if nans_posteriores > 0:
    print("\n⚠️ Nota Metodológica: Han quedado NaNs.")
    print("Esto significa que hay sensores que estuvieron apagados más de 6 horas seguidas.")
else:
    print("\n✅ Todos los NaNs fueron imputados correctamente sin sobrepasar el límite.")

# Comprobamos las nuevas dimensiones
print(f"\nDimensiones finales del dataset: {df_pivot.shape}")



Ejecutando Paso 3...
Total de filas con algún NaN ANTES de imputar: 827 de 18960
Total de filas con algun NaN TRAS imputar (limite 6h): 192

⚠️ Nota Metodológica: Han quedado NaNs.
Esto significa que hay sensores que estuvieron apagados más de 6 horas seguidas.

Dimensiones finales del dataset: (18960, 12)


# Paso 4: Feature Engineering Temporal

In [4]:
print("Ejecutando Paso 4...")

# Extraemos las caracteristicas directamente del indice temporal
df_pivot['hora'] = df_pivot.index.hour
df_pivot['dia_semana'] = df_pivot.index.dayofweek # Lunes=0, ..., Domingo=6
df_pivot['mes'] = df_pivot.index.month

# Creamos una variable binaria muy predictiva para modelos ML
# 1 si es Sábado (5) o Domingo (6), 0 si es Lunes a Viernes
df_pivot['es_finde'] = (df_pivot.index.dayofweek >= 5).astype(int)

# Verificaciones
print(f"Número total de columnas ahora: {df_pivot.shape[1]}")
print("\nMuestra de las nuevas variables (primeras 5 filas):")
display(df_pivot[['hora', 'dia_semana', 'mes', 'es_finde']].head(5))

Ejecutando Paso 4...
Número total de columnas ahora: 16

Muestra de las nuevas variables (primeras 5 filas):


,hora,dia_semana,mes,es_finde
fecha,,,,
2024-01-01 00:00:00,0,0,1,0
2024-01-01 01:00:00,1,0,1,0
2024-01-01 02:00:00,2,0,1,0
2024-01-01 03:00:00,3,0,1,0
2024-01-01 04:00:00,4,0,1,0


# Paso 5: Partición Física (Split Cronológico) y Exportación

In [6]:
print("Ejecutando Paso 5...")

# 1. Definir los limites temporales (fechas de corte)
train_end = '2025-10-31 23:00:00'
val_start = '2025-11-01 00:00:00'
val_end   = '2025-12-31 23:00:00'
test_start = '2026-01-01 00:00:00'

# 2. Hacer el split usando el indice temporal (.loc incluye los extremos)
df_train = df_pivot.loc[ : train_end ]
df_val   = df_pivot.loc[ val_start : val_end ]
df_test  = df_pivot.loc[ test_start : ]

# 3. Validaciones estrictas (Comprobaciones de seguridad estricta)
# Si alguna no cumple arrojamos mensaje
assert df_train.index.max() < df_val.index.min(), "❌ Error: Solapamiento entre Train y Val"
assert df_val.index.max() < df_test.index.min(), "❌ Error: Solapamiento entre Val y Test"

total_filas_split = len(df_train) + len(df_val) + len(df_test)
assert total_filas_split == len(df_pivot), "❌ Error: Se han perdido filas en el corte"

print("\n--- Verificación del Split ---")
print(f"✅ TRAIN: {len(df_train):>6} filas | {df_train.index.min()} -> {df_train.index.max()}")
print(f"✅ VAL:   {len(df_val):>6} filas | {df_val.index.min()} -> {df_val.index.max()}")
print(f"✅ TEST:  {len(df_test):>6} filas | {df_test.index.min()} -> {df_test.index.max()}")
print(f"\nSuma de particiones: {total_filas_split} filas. (Coincide con el total).")

# 4. Exportación a disco
df_train.to_csv('../data/processed/Split_Datasets/data_train.csv')
df_val.to_csv('../data/processed/Split_Datasets/data_val.csv')
df_test.to_csv('../data/processed/Split_Datasets/data_test.csv')
print("\n✅ Archivos exportados: 'data_train.csv', 'data_val.csv', 'data_test.csv'.")

Ejecutando Paso 5...

--- Verificación del Split ---
✅ TRAIN:  16080 filas | 2024-01-01 00:00:00 -> 2025-10-31 23:00:00
✅ VAL:     1464 filas | 2025-11-01 00:00:00 -> 2025-12-31 23:00:00
✅ TEST:    1416 filas | 2026-01-01 00:00:00 -> 2026-02-28 23:00:00

Suma de particiones: 18960 filas. (Coincide con el total).

✅ Archivos exportados: 'data_train.csv', 'data_val.csv', 'data_test.csv'.
